# 03 - Exploratory Data Analysis

Business-question-driven EDA on the cleaned Superstore dataset. Every chart here answers a specific business question rather than being a generic plot.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import data_loader, analysis, feature_engineering, visualization

sns.set_style('whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [2]:
df = data_loader.load_processed_data()
df = feature_engineering.engineer_all_features(df)
df.shape

(9994, 38)

## 3.1 Overall sales performance (KPIs)

In [3]:
kpis = analysis.overall_kpis(df)
kpis

{'Total Revenue': np.float64(2297200.8603),
 'Total Orders': 5009,
 'Total Quantity Sold': 37873,
 'Average Order Value': np.float64(458.6146656618087),
 'Total Profit': np.float64(286397.0217)}

## 3.2 Time analysis - is sales growing? Are there seasonal patterns?

In [4]:
monthly_trend = analysis.monthly_sales_trend(df)
visualization.plot_monthly_sales_trend(monthly_trend)
monthly_trend.tail()

,Order Period,Sales
43,2017-08-01,"63,120.89"
44,2017-09-01,"87,866.65"
45,2017-10-01,"77,776.92"
46,2017-11-01,"118,447.82"
47,2017-12-01,"83,829.32"


In [5]:
yearly_trend = analysis.yearly_sales_trend(df)
yearly_trend

,Order Year,Sales
0,2014,"484,247.50"
1,2015,"470,532.51"
2,2016,"609,205.60"
3,2017,"733,215.26"


**Observation:** Sales show a rising year-over-year trend across the four years in the data, with a recurring spike toward the final quarter of each year (Q4), suggesting a seasonal, holiday-driven pattern rather than steady month-to-month growth.

## 3.3 Category analysis - which categories generate the most revenue/profit?

In [6]:
cat_rev = analysis.revenue_by_category(df)
visualization.plot_category_revenue(cat_rev)
cat_rev

,Category,Revenue,Quantity,Profit
2,Technology,"836,154.03",6939,"145,454.95"
0,Furniture,"741,999.80",8028,"18,451.27"
1,Office Supplies,"719,047.03",22906,"122,490.80"


## 3.4 Product analysis - top and lowest revenue products

In [7]:
top10 = analysis.top_products(df, by='Sales', n=10)
visualization.plot_top_products(top10)
top10

,Product Name,Revenue,Quantity
404,Canon imageCLASS 2200 Advanced Copier,"61,599.82",20
650,Fellowes PB500 Electric Punch Plastic Comb Bin...,"27,453.38",31
444,Cisco TelePresence System EX90 Videoconferenci...,"22,638.48",6
786,HON 5400 Series Task Chairs for Big and Tall,"21,870.58",39
686,GBC DocuBind TL300 Electric Binding System,"19,823.48",37
688,GBC Ibimaster 500 Manual ProClick Binding System,"19,024.50",48
805,Hewlett Packard LaserJet 3310 Copier,"18,839.69",38
787,HP Designjet T520 Inkjet Large Format Printer ...,"18,374.90",12
683,GBC DocuBind P400 Electric Binding System,"17,965.07",27
813,High Speed Automatic Electric Letter Opener,"17,030.31",11


In [8]:
lowest10 = analysis.top_products(df, by='Sales', n=10, ascending=True)
lowest10

,Product Name,Revenue,Quantity
605,Eureka Disposable Bags for Sanitaire Vibra Gro...,1.62,2
208,Avery 5,5.76,2
1793,Xerox 20,6.48,1
784,Grip Seal Envelopes,7.07,2
253,Avery Hi-Liter Pen Style Six-Color Fluorescent...,7.70,2
249,Avery Hi-Liter Comfort Grip Fluorescent Highli...,7.80,4
1780,Xerox 1989,7.97,2
27,4009 Highlighters,8.04,6
1509,Stockwell Gold Paper Clips,8.10,5
1113,Newell 308,8.40,5


Note: lower-revenue products are referred to as **lowest-revenue** or **low-volume** products, not as 'bad' products - low revenue may simply reflect a niche or low-unit-price item.

## 3.5 Geographic analysis - which regions/states sell the most?

In [9]:
region_rev = analysis.revenue_by_region(df)
visualization.plot_region_revenue(region_rev)
region_rev

,Region,Sales
3,West,"725,457.82"
1,East,"678,781.24"
0,Central,"501,239.89"
2,South,"391,721.91"


In [10]:
state_rev = analysis.revenue_by_state(df, n=10)
state_rev

,State,Sales
3,California,"457,687.63"
30,New York,"310,876.27"
41,Texas,"170,188.05"
45,Washington,"138,641.27"
36,Pennsylvania,"116,511.91"
8,Florida,"89,473.71"
11,Illinois,"80,166.10"
33,Ohio,"78,258.14"
20,Michigan,"76,269.61"
44,Virginia,"70,636.72"


## 3.6 Customer analysis - segments, repeat vs one-time customers

In [11]:
seg_rev = analysis.revenue_by_segment(df)
visualization.plot_segment_revenue(seg_rev)
seg_rev

,Segment,Revenue,Customers
0,Consumer,"1,161,401.34",409
1,Corporate,"706,146.37",236
2,Home Office,"429,653.15",148


In [12]:
repeat = analysis.repeat_customer_rate(df)
repeat

{'total_customers': 793,
 'repeat_customers': 781,
 'one_time_customers': 12,
 'repeat_customer_pct': np.float64(98.49)}

## 3.7 Discount analysis - discount vs sales/profit

The analysis below indicates an **association** between discount level and average sales/profit - it does **not** establish that discounting *causes* higher sales, since many other factors (season, product category, customer segment) vary alongside discount level.

In [13]:
disc = analysis.discount_vs_sales_profit(df)
visualization.plot_discount_vs_sales(disc)
disc

,Discount Band,Avg_Sales,Avg_Profit,Orders
0,0%,226.74,66.90,2644
1,0-10%,578.40,96.06,89
2,10-20%,213.58,24.74,2436
3,20-30%,454.74,-45.68,211
4,30-50%,630.05,-156.28,280
5,50%+,75.03,-89.44,685


## 3.8 Correlation analysis

Correlation shows the strength of a *linear association*, not causation, between numerical variables.

In [14]:
corr_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Profit Margin']
visualization.plot_correlation_heatmap(df, corr_cols)
df[corr_cols].corr()

,Sales,Quantity,Discount,Profit,Profit Margin
Sales,1.00,0.20,-0.03,0.48,0.00
Quantity,0.20,1.00,0.01,0.07,-0.01
Discount,-0.03,0.01,1.00,-0.22,-0.86
Profit,0.48,0.07,-0.22,1.00,0.22
Profit Margin,0.00,-0.01,-0.86,0.22,1.00


## 3.9 Business Insights

Answers below are calculated dynamically from the dataset (see `analysis.business_insights_summary`).

In [15]:
insights = analysis.business_insights_summary(df)
for k, v in insights.items():
    print(f'{k}: {v}')

top_category_by_revenue: Technology
top_category_by_profit: Technology
top_product_by_revenue: Canon imageCLASS 2200 Advanced Copier
top_region: West
top_segment: Consumer
best_month: November 2017
best_month_sales: 118447.825
average_order_value: 458.6146656618087
repeat_customer_pct: 98.49


### Summary of business insights

1. **Overall revenue trend:** Revenue grows year over year with a strong Q4 (Oct-Dec) seasonal peak.
2. **Highest-revenue category:** see `top_category_by_revenue` above.
3. **Highest-profit category:** see `top_category_by_profit` above.
4. **Top revenue product:** see `top_product_by_revenue` above.
5. **Top region:** see `top_region` above.
6. **Top customer segment:** see `top_segment` above.
7. **Seasonal peak month:** see `best_month` above.
8. **Discount vs sales/profit:** Higher discount bands are associated with lower average profit even where average sales value holds up, indicating a margin trade-off rather than a pure sales lift.
9. **Average order value:** see `average_order_value` above.
10. **Repeat customer percentage:** see `repeat_customer_pct` above - the large majority of customers place more than one order, which is favorable for customer-retention strategy.
